In [21]:
import pandas as pd
import geopandas as gpd

# Load data
df = pd.read_csv('/Users/jackzipper/QSS20/final_project_data/iati-activity-locations-in-democratic-republic-of-the-congo.csv')
provinces = gpd.read_file('/Users/jackzipper/QSS20/final_project_data/cod_admin_boundaries.shp/cod_admin1.shp')

# Project to meters-based CRS for accurate distance calculations
provinces_proj = provinces.to_crs('EPSG:32635')

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['location_longitude'], df['location_latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

# Spatial join - exact match first
joined = gpd.sjoin(gdf, provinces_proj[['adm1_name', 'geometry']], how='left', predicate='within')
joined = joined.drop(columns=['index_right'])
joined['province'] = joined['adm1_name']

# For remaining NAs, use nearest province with max_distance of ~55km
na_mask = joined['province'].isna()
gdf_na = gpd.GeoDataFrame(
    joined[na_mask].drop(columns=['adm1_name']),
    geometry=gpd.points_from_xy(
        joined.loc[na_mask, 'location_longitude'],
        joined.loc[na_mask, 'location_latitude']
    ),
    crs='EPSG:4326'
).to_crs('EPSG:32635')

nearest = gpd.sjoin_nearest(gdf_na, provinces_proj[['adm1_name', 'geometry']], how='left', max_distance=55000)
joined.loc[nearest.index, 'province'] = nearest['adm1_name'].values

# Drop rows still NA (outside DRC) and clean up
df_final = joined.drop(columns=['adm1_name', 'geometry'])
df_final = df_final.dropna(subset=['province'])
df_final['location_name'] = df_final['province']
df_final = df_final.drop(columns=['province'])

# --- Deduplicate before saving ---
print(f"Before dedup: {len(df_final)} rows")

# Step 1: drop exact duplicates
df_final = df_final.drop_duplicates()
print(f"After dropping exact dupes: {len(df_final)} rows")

# Step 2: drop remaining dupes on key columns. Some columns have different longtiude and latitudes, 
# meaning that they are the same aid project in different locations in the same province. This would not get 
# picked up on by the drop_duplicates() method.
key_cols = ['aid', 'location_name', 'day_start', 'day_end', 'description', 'spend']
df_final = df_final.drop_duplicates(subset=key_cols)

# Save
df_final.to_csv('/Users/jackzipper/QSS20/final_project_data/iati-drc-cleaned.csv', index=False)

Before dedup: 28324 rows
After dropping exact dupes: 6025 rows
After dropping key col dupes: 4917 rows
Unique aid projects: 2836

NAs in location_name: 0
Unique provinces: 26
location_name
Kinshasa          1338
Nord-Kivu          445
Kasaï              434
Sud-Kivu           394
Sankuru            249
Ituri              229
Tanganyika         198
Haut-Katanga       174
Tshopo             173
Kasaï-Oriental     145
Maniema            134
Kasaï-Central      133
Kongo-Central      110
Lomami              85
Equateur            70
Lualaba             70
Haut-Lomami         70
Bas-Uele            70
Kwilu               69
Sud-Ubangi          62
Kwango              52
Nord-Ubangi         49
Haut-Uele           48
Maï-Ndombe          39
Tshuapa             39
Mongala             38
Name: count, dtype: int64
